# MA3001B • Semana 07: Atípicos, ARIMA/SARIMA y AIC/BIC
## Detección Robusta con MAD, Ciclo de Box-Jenkins e Inferencia con Intervalos
**Profesor B: Modelos y TDA • Tecnológico de Monterrey**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alxcn/Models_MA3001B/blob/main/notebooks/N4_Atipicos_ARIMA_SARIMA.ipynb)

---
### Objetivos de la Sesión
1. Detectar y tratar atípicos con un método robusto basado en la **mediana absoluta de las desviaciones (MAD)** sobre el residuo STL.
2. Formular y ajustar procesos **ARIMA** y **SARIMA** respetando el ciclo de Box-Jenkins.
3. Comparar modelos de forma parsimoniosa mediante criterios de información **AIC** y **BIC**.
4. Diagnosticar rigurosamente los residuos con la prueba **Ljung-Box** y normalidad visual.
5. Generar pronósticos con **intervalos de confianza probabilísticos al 95%**.


## 1. Problema Detonador: Picos Contaminantes y Pronóstico sin Incertidumbre
Sembraremos dos anomalías severas ($\pm 8$ ppm) en la serie. Observemos cómo distorsionan un modelo ordinario y cómo la gerencia demanda intervalos de confianza del 95% que las redes neuronales estándar no proveen.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.ar_model import ar_select_order
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

y_det = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
y_pico = y_det.copy()
# Sembrado artificial de anomalías
y_pico.iloc[[100, 250]] += [8.0, -8.0]

print(f"Valores alterados en t=100 y t=250: {y_pico.iloc[100]:.1f} y {y_pico.iloc[250]:.1f}")
print("-> Si ajustamos un modelo sin limpiar estas anomalías, los residuos no serán ruido blanco.")


## 2. Marco Teórico Mínimo

### 2.1 Regla Robusta de la MAD sobre Residuo STL
$$z_t = \frac{0.6745 \cdot (r_t - \operatorname{mediana}(r))}{\operatorname{MAD}(r)}, \qquad \text{Atípico si } |z_t| > 3.5$$

### 2.2 Modelo SARIMA Completo
$$\Phi_P(B^s) \phi_p(B) (1 - B)^d (1 - B^s)^D y_t = c + \Theta_Q(B^s) \theta_q(B) \varepsilon_t$$

### 2.3 Criterios de Selección: AIC vs. BIC
$$\operatorname{AIC} = 2k - 2\ln\hat{L}, \qquad \operatorname{BIC} = k\ln n - 2\ln\hat{L}$$


### Configuración de Datos: Práctica (CO₂) o Serie del Reto
En la siguiente celda se carga la serie de práctica por defecto (**CO₂ mensual de Mauna Loa** de `statsmodels`).
Para aplicar el análisis a la serie de su equipo en el Reto, cambie la bandera `USAR_SERIE_RETO = True` y proporcione la ruta a su archivo CSV.

In [ ]:
# ==============================================================================
# CELDA SERIE_RETO: Alternar entre datos de práctica y datos del Reto
# ==============================================================================
USAR_SERIE_RETO = False  # Cambiar a True para usar la serie de su Reto

if USAR_SERIE_RETO:
    # Sustituya con la carga de la serie de su equipo:
    # df_reto = pd.read_csv("datos_reto.csv", index_col=0, parse_dates=True)
    # y = df_reto["columna_objetivo"].asfreq("MS").interpolate()
    print("Modo Reto activado.")
else:
    # Serie de práctica por defecto: CO2 de Mauna Loa (mensual interpolado)
    y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
    print(f"Cargada serie de práctica CO2: {len(y)} observaciones ({y.index.min().strftime('%Y-%m')} a {y.index.max().strftime('%Y-%m')})")


## 3. Ejemplo Aplicado Paso a Paso

### Paso 1: Detección Robusta de Atípicos y Enmascaramiento NaN

In [ ]:
# 1) Descomposición STL robusta
stl_clean = STL(y, period=12, robust=True).fit()
resid = stl_clean.resid

# 2) Puntuación Z modificada por MAD
mediana_r = resid.median()
mad_r = np.median(np.abs(resid - mediana_r))
z_mad = 0.6745 * (resid - mediana_r) / mad_r

atipicos_idx = y.index[np.abs(z_mad) > 3.5]
print(f"Número de atípicos detectados con umbral |z| > 3.5: {len(atipicos_idx)}")
if len(atipicos_idx) > 0:
    print(f"Fechas de atípicos: {atipicos_idx.strftime('%Y-%m').tolist()}")

# Enmascarar con NaN (SARIMAX maneja NaN con el filtro de Kalman automáticamente)
y_limpia = y.mask(np.abs(z_mad) > 3.5)


### Paso 2: Selección de Rezagos AR Puros

In [ ]:
dy = y_limpia.diff().dropna()
ar_sel = ar_select_order(dy, maxlag=24, ic="aic", trend="c")
print(f"Rezagos autorregresivos sugeridos por AIC en la serie diferenciada: {ar_sel.ar_lags}")


### Paso 3: Rejilla Sistemática de Modelos SARIMA (d=1, D=1, s=12)

In [ ]:
# Partición cronológica: últimos 24 meses para prueba
train_s = y_limpia.iloc[:-24]
test_s = y_limpia.iloc[-24:]

filas_grid = []
# Rejilla reducida y rápida: p in [0,1,2], q in [0,1,2], P in [0,1], Q in [0,1]
for p, q, P, Q in itertools.product(range(3), range(3), range(2), range(2)):
    try:
        mod_iter = SARIMAX(
            train_s,
            order=(p, 1, q),
            seasonal_order=(P, 1, Q, 12),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
        filas_grid.append({
            "order": (p, 1, q),
            "seasonal_order": (P, 1, Q, 12),
            "AIC": mod_iter.aic,
            "BIC": mod_iter.bic
        })
    except Exception:
        continue

df_grid = pd.DataFrame(filas_grid).sort_values("BIC")
print("=== TOP 5 MODELOS SARIMA ORDENADOS POR BIC ===")
print(df_grid.head().to_string(index=False))

mejor_modelo_cfg = df_grid.iloc[0]
print(f"\n-> Mejor modelo seleccionado: order={mejor_modelo_cfg['order']}, seasonal={mejor_modelo_cfg['seasonal_order']}")


### Paso 4: Ajuste del Mejor SARIMA, Diagnóstico y Prueba Ljung-Box

In [ ]:
mejor_sarima = SARIMAX(
    train_s,
    order=mejor_modelo_cfg["order"],
    seasonal_order=mejor_modelo_cfg["seasonal_order"],
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(mejor_sarima.summary().tables[1])

# Diagnóstico de residuos
fig = mejor_sarima.plot_diagnostics(figsize=(10, 6))
plt.tight_layout()
plt.show()

# Prueba formal de Ljung-Box sobre los residuos
lb_res = acorr_ljungbox(mejor_sarima.resid.dropna().iloc[13:], lags=[12, 24], return_df=True)
print("\n=== PRUEBA LJUNG-BOX SOBRE RESIDUOS ===")
print(lb_res)


### Paso 5: Pronóstico con Bandas de Confianza al 95% y Evaluación

In [ ]:
fc_frame = mejor_sarima.get_forecast(steps=24).summary_frame(alpha=0.05)

plt.figure(figsize=(10, 4.5))
plt.plot(train_s.index[-36:], train_s.iloc[-36:], label="Histórico Entrenamiento", color="#073642")
plt.plot(test_s.index, test_s, label="Real Prueba", color="#2AA198", lw=2)
plt.plot(fc_frame.index, fc_frame["mean"], label="Pronóstico SARIMA", color="#CB4B16", linestyle="--", lw=2)
plt.fill_between(fc_frame.index, fc_frame["mean_ci_lower"], fc_frame["mean_ci_upper"], color="#CB4B16", alpha=0.2, label="Intervalo Confianza 95%")

plt.title(f"Pronóstico SARIMA {mejor_modelo_cfg['order']}x{mejor_modelo_cfg['seasonal_order']} con Bandas 95%")
plt.xlabel("Fecha")
plt.ylabel("CO2 (ppm)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

mae_sarima = np.mean(np.abs(test_s.values - fc_frame["mean"].values))
rmse_sarima = np.sqrt(np.mean((test_s.values - fc_frame["mean"].values)**2))
print(f"Métricas en Conjunto de Prueba (h=24 meses):")
print(f"MAE SARIMA:  {mae_sarima:.4f} ppm")
print(f"RMSE SARIMA: {rmse_sarima:.4f} ppm")


## 4. Actividad del Reto y Entregable
1. Aplique la detección de atípicos por MAD en su serie del Reto.
2. Construya una rejilla de al menos 12 combinaciones SARIMA y elija el modelo óptimo con BIC.
3. Reporte el pronóstico a su horizonte de interés con su respectiva banda de incertidumbre al 95%.

In [ ]:
# ==============================================================================
# TODO 1: Rejilla SARIMA sobre la serie del Reto
# ==============================================================================
# Ajuste la rejilla y ordene por BIC.


In [ ]:
# ==============================================================================
# TODO 2: Diagnóstico Ljung-Box y Gráficas de Residuos
# ==============================================================================
# Valide que los residuos sean ruido blanco (p-valor > 0.05).


## 5. Conclusiones y Pregunta Abierta
- **Conclusión:** SARIMA entrega una solución parsimoniosa, estadísticamente interpretable y con bandas de probabilidad cuantitativas indispensables para la toma de decisiones.
- **Pregunta para Semana 08:** Si necesitamos un pronóstico actualizable en milisegundos con cada nueva observación, ¿cómo compite el suavizado exponencial (Holt-Winters) y quién triunfa en el benchmark integrador final?
